In [1]:
from typing_extensions import  TypedDict
from langgraph.graph import START,END,StateGraph
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

ModuleNotFoundError: No module named 'langgraph.graph'

In [ ]:
class SubState(StateGraph):
    input_text:str
    translated_text:str

In [ ]:
subgraph_llm=ChatGroq(model="")

In [ ]:
def translate_text(state:SubState):
    prompt=f"""
    Translate the following text to Hindi.Keep ie natural and clear.Do not add extra content.
    text:{state.input_text}
    """.strip()
    tarnslated_text=subgraph_llm.invoke(prompt)
    return {
        "translated_text":translated_text
    }

In [ ]:
subgraph_builder=StateGraph(SubState)

subgraph_builder.add_node("translate_text",translate_text)

subgraph_builder.add_edge(START,"translate_text")
subgraph_builder.add_edge("translate_text",END)
subgraph=subgraph_builder.compile()
subgraph

In [ ]:
class ParentState(TypedDict):
    question:str
    answer_eng:str
    answer_hin:str

In [ ]:
parent_llm=ChatGroq(model="")

In [ ]:
def generate_answer(state:ParentState):
    answer=parent_llm.invoke(f"You are helpful assistant.Answer clearly.\n Question:{state['question']}").content
    return {
        "answer_eng":answer
    }

In [ ]:
def translate_answer(state:ParentState):
    result=subgraph.invoke({'input_text':state['answer_eng']})
    return {
        "answer_hin":result['translated_text']
    }

In [ ]:
parent_builder=StateGraph(ParentState)
parent_builder.add_node("answer",generate_answer)
parent_builder.add_node("translate",translate_answer)

parent_builder.add_edge(START,"answer")
parent_builder.add_edge("answer","translate")
parent_builder.add_edge("translate",END)

graph=parent_builder.compile()
graph

In [ ]:
graph.invoke({'question':"How are you?"})